In [27]:
from typing import Optional

In [ ]:
class Problema_Baldes:

# OPERAÇÕES INTERNAS
    def __init__(self, baldes: Optional[list[int]] = None):
        self.baldes: list[int] = baldes if baldes is not None else [5, 0]

    def _get_balde_maior(self) -> int:
        return self.baldes[0]

    def _get_balde_menor(self) -> int:
        return self.baldes[1]

    def _set_balde_maior(self, valor: int) -> bool:
        if valor > 5 or valor < 0:
            return False
        self.baldes[0] = valor
        return True

    def _set_balde_menor(self, valor: int) -> bool:
        if valor > 3 or valor < 0:
            return False
        self.baldes[1] = valor
        return True

# OPERAÇÕES AUXILIARES
    def clonar(self) -> "Problema_Baldes":
        return Problema_Baldes(self.baldes.copy())

    def tupla(self) -> tuple[int, int]:
        return (self.baldes[0], self.baldes[1])

    def print(self):
        print(f'BALDE MAIOR: {self.baldes[0]}')
        print(f'BALDE MENOR: {self.baldes[1]}')

    def is_solucionado(self) -> bool:
        return (self._get_balde_menor() + self._get_balde_maior()) == 4

# REGRAS DE TRANSIÇÃO
    def enche_balde_maior(self) -> bool:
        valor = self._get_balde_maior()
        if valor >= 5:
            return False
        self._set_balde_maior(5)
        return True

    def enche_balde_menor(self) -> bool:
        valor = self._get_balde_menor()
        if valor >= 3:
            return False
        self._set_balde_menor(3)
        return True

    def esvazia_balde_maior(self) -> bool:
        valor = self._get_balde_maior()
        if valor < 1:
            return False
        self._set_balde_maior(0)
        return True

    def esvazia_balde_menor(self) -> bool:
        valor = self._get_balde_menor()
        if valor < 1:
            return False
        self._set_balde_menor(0)
        return True

    def balde_maior_to_balde_menor(self) -> bool:
        valor_maior = self._get_balde_maior()
        valor_menor = self._get_balde_menor()

        if valor_maior == 0 or valor_menor >= 3:
            return False

        folga_menor = 3 - valor_menor
        if valor_maior <= folga_menor:
            self._set_balde_menor(valor_menor + valor_maior)
            self._set_balde_maior(0)
        else:
            self._set_balde_menor(3)
            self._set_balde_maior(valor_maior - folga_menor)
        return True

    def balde_menor_to_balde_maior(self) -> bool:
        valor_maior = self._get_balde_maior()
        valor_menor = self._get_balde_menor()

        if valor_menor == 0 or valor_maior >= 5:
            return False

        folga_maior = 5 - valor_maior
        if valor_menor <= folga_maior:
            self._set_balde_maior(valor_menor + valor_maior)
            self._set_balde_menor(0)
        else:
            self._set_balde_maior(5)
            self._set_balde_menor(valor_menor - folga_maior)
        return True

In [29]:
class Estado_Busca:

    ORDEM_EXECUCAO = [
        "regra_5",
        "regra_2",
        "regra_4",
        "regra_3",
        "regra_1",
        "regra_6",
    ]
    
    def __init__(self, pai: Optional["Estado_Busca"] = None, baldes: Problema_Baldes = None):
        self.baldes = baldes if baldes else Problema_Baldes()
        self.pai = pai
        self.proxima_regra = 0
        self.impasse = False
        self.regras = {
            "regra_1": self.baldes.enche_balde_menor,
            "regra_2": self.baldes.enche_balde_maior,
            "regra_3": self.baldes.esvazia_balde_menor,
            "regra_4": self.baldes.esvazia_balde_maior,
            "regra_5": self.baldes.balde_menor_to_balde_maior,
            "regra_6": self.baldes.balde_maior_to_balde_menor,
        }
        

    def gerar_filho(self):
        while self.proxima_regra < len(self.ORDEM_EXECUCAO):
            nome_regra = self.ORDEM_EXECUCAO[self.proxima_regra]
            self.proxima_regra += 1
            
            candidato = Estado_Busca(pai=self, baldes=self.baldes.clonar())
            if candidato.regras[nome_regra]():
                return candidato
        self.impasse = True
        return None

In [30]:
class Busca_Baldes:
    def __init__(self):
        self.baldes = Problema_Baldes()
        self.nivel = 0
        self.estados = [Estado_Busca(pai=None, baldes=self.baldes)]
        self.visitados = {self.baldes.tupla()}

    def print(self):
        self.estados[self.nivel].baldes.print()

    def busca_completa(self) -> bool:
        while not self.estados[self.nivel].baldes.is_solucionado():
            estado_atual = self.estados[self.nivel]
            novo_estado = estado_atual.gerar_filho()

            if novo_estado is not None:
                chave = (novo_estado.baldes.tupla())
                if chave not in self.visitados:
                    self.visitados.add(chave)
                    self.nivel += 1
                    self.estados.append(novo_estado)
            else:
                if self.nivel == 0:
                    return False
                self.estados.pop()
                self.nivel -= 1

        return True

    def imprime_solucao(self):
        caminho = []
        estado = self.estados[self.nivel]
        while estado is not None:
            caminho.append(estado)
            estado = estado.pai
        caminho.reverse()

        for i, estado in enumerate(caminho):
            print(f'--- Passo {i} ---')
            estado.baldes.print()

In [31]:
if __name__ == "__main__":
    busca = Busca_Baldes()
    if busca.busca_completa():
        print("Solução encontrada!\n")
        busca.imprime_solucao()
    else:
        print("Não foi possível encontrar solução (impasse).")

Solução encontrada!

--- Passo 0 ---
BALDE MAIOR: 5
BALDE MENOR: 0
--- Passo 1 ---
BALDE MAIOR: 0
BALDE MENOR: 0
--- Passo 2 ---
BALDE MAIOR: 0
BALDE MENOR: 3
--- Passo 3 ---
BALDE MAIOR: 3
BALDE MENOR: 0
--- Passo 4 ---
BALDE MAIOR: 3
BALDE MENOR: 3
--- Passo 5 ---
BALDE MAIOR: 5
BALDE MENOR: 1
--- Passo 6 ---
BALDE MAIOR: 0
BALDE MENOR: 1
--- Passo 7 ---
BALDE MAIOR: 1
BALDE MENOR: 0
--- Passo 8 ---
BALDE MAIOR: 1
BALDE MENOR: 3
